# Movielens - SVD

In [1]:
import pandas as pd
import numpy as np
import scipy

In [2]:
df = pd.read_csv("/Users/mauliksindhva/Downloads/movie-lens-1m/ratings.csv")
df.sample(10)

,user_id,movie_id,rating,timestamp
706813,4238,2051,2,965417262
508894,3129,1246,4,969293650
772304,4605,3105,3,964204191
276174,1671,2859,4,974713778
219084,1329,2478,4,978557146
166152,1058,1982,3,974957047
430765,2625,3519,4,973634878
71968,482,440,4,976221965
577901,3529,457,3,966898565
457956,2820,2146,2,972660256


In [3]:
df_movies = pd.read_csv("/Users/mauliksindhva/Downloads/movie-lens-1m/movies.csv")
df_movies.sample(10)

,movie_id,title,genres
166,168,First Knight (1995),Action|Adventure|Drama|Romance
1075,1091,Weekend at Bernie's (1989),Comedy
1810,1879,"Hanging Garden, The (1997)",Drama
28,29,"City of Lost Children, The (1995)",Adventure|Sci-Fi
2836,2905,Sanjuro (1962),Action|Adventure
3226,3295,Raining Stones (1993),Drama
966,978,"Blue Angel, The (Blaue Engel, Der) (1930)",Drama
2687,2756,Wanted: Dead or Alive (1987),Action
2288,2357,Central Station (Central do Brasil) (1998),Drama
2324,2393,Star Trek: Insurrection (1998),Action|Sci-Fi


## Pivot df to Matrix

In [4]:
df_pivot = df.pivot(index="user_id", columns="movie_id", values="rating")
df_pivot.shape

(6040, 3706)

In [5]:
df_pivot.sample(5)

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4503,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4007,4.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,3.0,...,NaN,NaN,1.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN
3157,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
578,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN


## Calculating Sparsness of Dataframe

In [6]:
1 - df.shape[0] / (len(df["user_id"].unique()) * len(df["movie_id"].unique()))

0.9553163743776871

## Removing Rating Bias

In [7]:
## Calculate the average ratings of each user
## deduct the average ratings from every rating by that user
## result: 0 means average, +: better than average, -: worse than average
## Calculating SVD
## Reconstructing Original Matrix
## Adding the mean users' average ratings

In [8]:
users_avg_rating = df_pivot.mean(axis=1)
len(users_avg_rating)

6040

In [9]:
df_norm = df_pivot.sub(users_avg_rating, axis='rows')
df_norm.sample(5)

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
335,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.5,NaN,1.5,NaN,NaN,NaN,-1.5
777,0.420319,0.420319,NaN,NaN,-0.579681,NaN,-0.579681,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1752,1.165899,NaN,NaN,NaN,NaN,0.165899,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.165899,NaN,NaN,NaN
2553,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_norm = df_norm.fillna(0)
df_norm.sample(5)

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
382,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
5201,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1827,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.408163
1921,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.258824
2340,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.360902,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [11]:
norm_matrix = df_norm.to_numpy()
u, s, vt = scipy.sparse.linalg.svds(norm_matrix, k=50)

u.shape, s.shape, vt.shape

((6040, 50), (50,), (50, 3706))

In [12]:
# reconstruct the matrix

regenerated = np.dot(u, np.dot(np.diag(s), vt))
regenerated.shape          

(6040, 3706)

In [13]:
df_predicted = pd.DataFrame(data=regenerated,
                            columns=df_pivot.columns,
                            index=df_pivot.index)
df_predicted.sample(5)

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
4787,-0.033978,0.001186,0.010800,-0.031911,-0.008761,-0.054373,-0.010392,-0.009444,0.011624,0.008097,...,-0.004820,0.006231,0.011372,0.004964,-0.001593,0.007582,-0.001858,0.001769,0.002344,-0.005152
1519,0.174547,-0.034167,0.037506,-0.058502,0.017633,0.028939,0.027389,0.024792,0.027347,0.006680,...,-0.039701,-0.005336,0.003764,-0.009115,-0.017175,-0.120225,0.012032,0.015369,0.044078,-0.024960
2676,-0.157662,-0.020699,-0.008914,-0.040940,-0.014472,-0.041370,0.010633,-0.009427,0.009154,-0.002820,...,0.002292,0.009217,-0.000934,0.003825,-0.008018,-0.028518,-0.006795,-0.006281,0.008156,-0.013042
5134,0.459706,0.010795,-0.079987,0.052582,0.057284,0.070419,0.126123,-0.001280,0.007284,-0.043317,...,-0.055268,0.000940,-0.020999,-0.061343,0.007373,0.114208,0.022623,-0.001192,0.012876,-0.015646
819,0.007728,-0.030336,0.115631,-0.017452,0.055377,-0.060510,0.040156,0.006720,0.015273,0.025027,...,-0.008977,0.012415,0.004733,0.023058,-0.004975,0.067511,-0.017249,-0.000469,-0.010891,-0.013211


In [14]:
df_predicted = df_predicted.add(users_avg_rating, axis='rows')
df_predicted.sample(5)

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1068,3.420254,3.685226,3.587518,3.581706,3.707292,4.004497,3.712186,3.735026,3.685940,3.935163,...,3.849101,3.777418,3.802319,3.740839,3.692768,3.939066,3.931253,3.703061,3.743094,3.802428
2457,3.945002,3.310879,3.369477,3.072114,3.212375,3.358449,3.245130,3.155357,3.079437,3.828995,...,3.162215,3.089358,2.947968,3.048211,3.137902,3.181385,3.089478,3.232088,3.223998,3.092224
1286,3.393344,3.449015,3.514679,3.522979,3.495717,3.544473,3.529748,3.489349,3.487403,3.493659,...,3.521069,3.502267,3.534233,3.531944,3.501943,3.418166,3.539689,3.497741,3.516033,3.518309
330,4.002752,3.958155,3.970247,3.989409,4.004263,4.024206,4.014002,3.997016,4.024381,4.042639,...,3.992041,4.003226,3.975760,3.982800,4.010753,3.891498,4.061729,4.020871,4.024928,4.028589
1088,3.350639,2.350495,3.074937,2.975484,2.925126,3.758549,3.146339,3.105288,3.089133,3.854717,...,3.313354,3.194385,2.988135,2.870537,3.378182,3.310426,4.127522,3.419640,3.372806,3.668786


## Test

In [15]:
user_id = 5001
K = 10

df_user = df[df["user_id"] == user_id]
df_user

,user_id,movie_id,rating,timestamp
831852,5001,588,5,1019973625
831853,5001,2,4,1019973871
831854,5001,7,4,1019973149
831855,5001,2062,3,1019973199
831856,5001,593,4,962929057
...,...,...,...,...
832072,5001,551,5,1019973649
832073,5001,1088,5,1019973023
832074,5001,555,1,1019973023
832075,5001,1094,4,1019972926


In [16]:
user_watched_movies = df_user["movie_id"].to_list()
len(user_watched_movies)

225

In [17]:
df_user_prediction = df_predicted[df_predicted.index == user_id]
df_user_prediction

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
5001,4.149496,4.168979,4.029166,4.084364,4.143398,4.04055,4.151308,4.113389,4.115482,4.12115,...,4.099076,4.1152,4.097653,4.108887,4.0936,4.109018,3.971802,4.072889,4.069329,4.105001


In [18]:
df_user_prediction = df_user_prediction.T.reset_index().sort_values(by=[user_id], ascending=False)
df_user_prediction.head(10)

user_id,movie_id,5001
2374,2571,4.876450
2455,2657,4.816351
1107,1197,4.781089
2203,2396,4.694297
970,1035,4.633122
963,1028,4.578506
1281,1380,4.546570
1003,1073,4.520667
859,920,4.476800
3341,3578,4.448429


In [19]:
n = 0
recommendations = []
for index, row in df_user_prediction.iterrows():
    if row["movie_id"] not in user_watched_movies:
        recommendations.append(int(row["movie_id"]))
        n += 1
        if n == K:
            break

len(recommendations)

10

In [20]:
recommendations

[3578, 597, 39, 608, 595, 899, 745, 1285, 1947, 3176]

## User Watched

In [21]:
df_user.merge(df_movies, on="movie_id", how="inner")

,user_id,movie_id,rating,timestamp,title,genres
0,5001,588,5,1019973625,Aladdin (1992),Animation|Children's|Comedy|Musical
1,5001,2,4,1019973871,Jumanji (1995),Adventure|Children's|Fantasy
2,5001,7,4,1019973149,Sabrina (1995),Comedy|Romance
3,5001,2062,3,1019973199,"Governess, The (1998)",Drama|Romance
4,5001,593,4,962929057,"Silence of the Lambs, The (1991)",Drama|Thriller
...,...,...,...,...,...,...
220,5001,551,5,1019973649,"Nightmare Before Christmas, The (1993)",Children's|Comedy|Musical
221,5001,1088,5,1019973023,Dirty Dancing (1987),Musical|Romance
222,5001,555,1,1019973023,True Romance (1993),Action|Crime|Romance
223,5001,1094,4,1019972926,"Crying Game, The (1992)",Drama|Romance|War


In [22]:
df_movies[df_movies["movie_id"].isin(recommendations)]

,movie_id,title,genres
38,39,Clueless (1995),Comedy|Romance
591,595,Beauty and the Beast (1991),Animation|Children's|Musical
593,597,Pretty Woman (1990),Comedy|Romance
604,608,Fargo (1996),Crime|Drama|Thriller
735,745,"Close Shave, A (1995)",Animation|Comedy|Thriller
887,899,Singin' in the Rain (1952),Musical|Romance
1265,1285,Heathers (1989),Comedy
1878,1947,West Side Story (1961),Musical|Romance
3107,3176,"Talented Mr. Ripley, The (1999)",Drama|Mystery|Thriller
3509,3578,Gladiator (2000),Action|Drama
